<a href="https://colab.research.google.com/github/cloudmrhub/camrie-tools/blob/v1/camrie_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CAMRIE Tools — Colab Guide

This notebook installs the full `camrie-tools` stack, checks the installation, and runs a small functionality example.

The repository/package follows the CAMRIE app family naming convention:

- `camrie` — main app
- `camrie-webgui` — frontend
- `camrie-tools` — Python package distribution
- `camrie-app` — cloud builder

Python imports use `camrie_tools` because Python module names cannot contain hyphens.

Use a GPU runtime if you want GPU execution. The package can still install on CPU-only runtimes; `CUDA.functional()` reports whether GPU execution is actually available.

## 1. Install Julia

Colab Python runtimes usually do not include Julia, so install it first.

In [ ]:
!curl -fsSL https://install.julialang.org | sh -s -- -y

In [ ]:
import os

os.environ["PATH"] = f"{os.path.expanduser('~')}/.juliaup/bin:" + os.environ["PATH"]
!julia --version

## 2. Install camrie-tools

In [ ]:
%pip install git+https://github.com/cloudmrhub/camrie-tools@v1

In [ ]:
import camrie_tools

print("camrie_tools", camrie_tools.__version__)
print("Bundled Julia script:", camrie_tools.simulate_batch_path())

## 3. Install the full Julia dependency stack

This installs `KomaInterface.jl` into the CAMRIE Julia project. The notebook uses `--cpu` because it works reliably on CPU-only Colab runtimes. On a CUDA-capable runtime, you can remove `--cpu` to also install `CUDA.jl`.

In [ ]:
!camrie-install-julia --cpu

## 4. Verify, run examples, and show a reconstruction

In [ ]:
!camrie-test-installation --cpu

In [ ]:
!camrie-example

In [ ]:
# ============================================================
# CAMRIE reconstruction smoke test and visualization
# ============================================================

from pathlib import Path
import json

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, Image as IPythonImage

from camrie_tools._reconstruction_smoke import run_reconstruction_smoke


# ------------------------------------------------------------
# 1. Configure the output directory
# ------------------------------------------------------------

output_dir = Path("/content/camrie_reconstruction_smoke")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Matplotlib backend: {matplotlib.get_backend()}")
print(f"Output directory: {output_dir}")


# ------------------------------------------------------------
# 2. Run the CAMRIE reconstruction smoke test
# ------------------------------------------------------------

summary = run_reconstruction_smoke(
    output_dir=str(output_dir),
    n_threads=1,
    grid_size=151,
    radius_mm=45.0,
)


# ------------------------------------------------------------
# 3. Print the main test results
# ------------------------------------------------------------

test_results = {
    "output_dir": str(output_dir),
    "spin_count": summary["spin_count"],
    "kspace_shape": summary["kspace_shape"],
    "reconstruction_shape": summary["reconstruction_shape"],
    "peak": summary["peak"],
}

print("\nCAMRIE reconstruction smoke-test results:")
print(json.dumps(test_results, indent=2))


# ------------------------------------------------------------
# 4. Locate and validate the reconstruction
# ------------------------------------------------------------

reconstruction_path = Path(
    summary["outputs"]["reconstruction_magnitude"]
)

if not reconstruction_path.exists():
    raise FileNotFoundError(
        "The reconstruction file was not created:\n"
        f"{reconstruction_path}"
    )

recon = np.load(reconstruction_path)

if recon.ndim != 2:
    raise ValueError(
        f"Expected a two-dimensional reconstruction, got shape {recon.shape}."
    )

if not np.all(np.isfinite(recon)):
    raise ValueError(
        "The reconstruction contains NaN or infinite values."
    )

recon_min = float(np.min(recon))
recon_max = float(np.max(recon))
recon_mean = float(np.mean(recon))

print("\nReconstruction validation:")
print(f"  File: {reconstruction_path}")
print(f"  Shape: {recon.shape}")
print(f"  Minimum: {recon_min:.6g}")
print(f"  Maximum: {recon_max:.6g}")
print(f"  Mean: {recon_mean:.6g}")

if recon_max <= 0:
    raise ValueError(
        "The reconstruction is empty or has no positive signal."
    )


# ------------------------------------------------------------
# 5. Create and save the visualization
# ------------------------------------------------------------

preview_path = output_dir / "reconstruction_preview.png"

fig, ax = plt.subplots(figsize=(6, 6))

image = ax.imshow(
    recon,
    cmap="gray",
    origin="lower",
)

ax.set_title(
    "CAMRIE circular phantom reconstruction\n"
    f"{summary['spin_count']} spins"
)
ax.set_xlabel("Readout")
ax.set_ylabel("Phase encoding")

colorbar = fig.colorbar(
    image,
    ax=ax,
    fraction=0.046,
    pad=0.04,
)

colorbar.set_label("Reconstruction magnitude")

fig.tight_layout()

fig.savefig(
    preview_path,
    dpi=160,
    bbox_inches="tight",
)

print(f"\nSaved reconstruction preview: {preview_path}")


# ------------------------------------------------------------
# 6. Display reliably in Colab/Jupyter
# ------------------------------------------------------------

display(fig)
plt.close(fig)

# This fallback displays the saved image even if Matplotlib uses Agg.
display(IPythonImage(filename=str(preview_path)))

In [ ]:
# Import the pipeline module when the runtime dependencies are installed.
from camrie_tools.MRI_pipeline import read_pulseq_params

print(read_pulseq_params)